In [50]:
from dotenv import load_dotenv
load_dotenv()

True

In [51]:
from langchain_community.document_loaders import TextLoader
from langchain.schema import Document

loader = TextLoader("transcript2.txt")
documents = loader.load()
pattern = r"\[(\d+\.\d+s)\]:\s*(.*)"

import re
docs = []
for doc in documents:
    for line in doc.page_content.splitlines():
        match = re.match(pattern, line)
        if match:
            timestamp = match.group(1)  # Extract the timestamp
            timestamp = timestamp[:-1]
            text = match.group(2)  # Extract the text
            if text.strip():  # Only process non-empty lines
                # Create a document with the text and metadata (timestamp)
                docs.append(text)


In [52]:
import operator
from typing import Annotated, List, TypedDict
from langchain.chains.summarize import load_summarize_chain
from langgraph.graph import END, START, StateGraph
from langchain.schema import Document
from langchain_google_genai import ChatGoogleGenerativeAI
import asyncio

In [53]:
# Set up the language model and chains

llm = ChatGoogleGenerativeAI(
    model = 'gemini-1.5-flash',
    temperature = 0,
    max_tokens = None,
    timeout = None,
    max_retries = 2
)
map_chain = load_summarize_chain(llm, chain_type="map_reduce")
reduce_chain = load_summarize_chain(llm, chain_type="map_reduce")
# Define the maximum token limit
token_max = 1000

In [54]:
# def length_function(documents: List[Document]) -> int:
#     """Calculate the total number of tokens for a list of documents."""
#     return sum(llm.get_num_tokens(doc.page_content) for doc in documents)

In [55]:
# Define data structures
class ChildData(TypedDict):
    text: str
class SubheadingData(TypedDict):
    subheading: str
    children: List[ChildData]
class HeadingData(TypedDict):
    heading: str
    children: List[SubheadingData]
class SummaryData(TypedDict):
    headings: List[HeadingData]

In [56]:
# Define the overall state
class OverallState(TypedDict):
    documents: List[Document]
    child_summaries: Annotated[List[ChildData], operator.add]
    subheading_summaries: Annotated[List[SubheadingData], operator.add]
    heading_summaries: Annotated[List[HeadingData], operator.add]
    final_summary: SummaryData

In [57]:
# Function to pad documents
def pad_documents(documents: List[str], total_length: int = 306) -> List[Document]:
    current_length = len(documents)
    padding_needed = total_length - current_length
    pad_each_side = padding_needed // 2
    padded_documents = (
        [''] * pad_each_side + documents + [''] * (padding_needed - pad_each_side)
    )
    return [Document(page_content=doc) for doc in padded_documents]

In [58]:
# Generate child summaries
async def generate_child_summaries(state: OverallState):
    num_children = 18
    total_docs = len(state['documents'])
    base_docs_per_child = total_docs // num_children
    remainder = total_docs % num_children
    index = 0
    state['child_summaries'] = []
    for i in range(num_children):
        docs_in_this_child = base_docs_per_child + (1 if i < remainder else 0)
        child_docs = state['documents'][index:index + docs_in_this_child]
        index += docs_in_this_child
        combined_text = ' '.join(doc.page_content for doc in child_docs)
        summary = await map_chain.arun(combined_text)
        state['child_summaries'].append({'text': summary})
    return state

In [59]:
# Generate subheading summaries
async def generate_subheading_summaries(state: OverallState):
    num_subheadings = 6
    total_children = len(state['child_summaries'])
    base_children_per_subheading = total_children // num_subheadings
    remainder = total_children % num_subheadings
    index = 0
    state['subheading_summaries'] = []
    for i in range(num_subheadings):
        children_in_this_subheading = base_children_per_subheading + (1 if i < remainder else 0)
        subheading_children = state['child_summaries'][index:index + children_in_this_subheading]
        index += children_in_this_subheading
        combined_text = ' '.join(child['text'] for child in subheading_children)
        summary = await reduce_chain.arun(combined_text)
        state['subheading_summaries'].append({
            'subheading': summary,
            'children': subheading_children
        })
    return state

In [60]:
# Generate heading summaries
async def generate_heading_summaries(state: OverallState):
    num_headings = 3
    total_subheadings = len(state['subheading_summaries'])
    base_subheadings_per_heading = total_subheadings // num_headings
    remainder = total_subheadings % num_headings
    index = 0
    state['heading_summaries'] = []
    for i in range(num_headings):
        subheadings_in_this_heading = base_subheadings_per_heading + (1 if i < remainder else 0)
        heading_subheadings = state['subheading_summaries'][index:index + subheadings_in_this_heading]
        index += subheadings_in_this_heading
        combined_text = ' '.join(sub['subheading'] for sub in heading_subheadings)
        summary = await reduce_chain.arun(combined_text)
        state['heading_summaries'].append({
            'heading': summary,
            'children': heading_subheadings
        })
    return state

In [61]:
# Generate final summary data structure
async def generate_final_summary(state: OverallState):
    state['final_summary'] = {'headings': state['heading_summaries']}
    return state

In [62]:
# Set up the state graph
graph = StateGraph(OverallState)
graph.add_node('generate_child_summaries', generate_child_summaries)
graph.add_node('generate_subheading_summaries', generate_subheading_summaries)
graph.add_node('generate_heading_summaries', generate_heading_summaries)
graph.add_node('generate_final_summary', generate_final_summary)
graph.add_edge(START, 'generate_child_summaries')
graph.add_edge('generate_child_summaries', 'generate_subheading_summaries')
graph.add_edge('generate_subheading_summaries', 'generate_heading_summaries')
graph.add_edge('generate_heading_summaries', 'generate_final_summary')
graph.add_edge('generate_final_summary', END)
app = graph.compile()

In [63]:
# Function to run the summarization
async def run_summarization(documents: List[Document]) -> SummaryData:
    padded_docs = pad_documents(documents)
    initial_state = OverallState(
        documents=padded_docs,
        child_summaries=[],
        subheading_summaries=[],
        heading_summaries=[],
        final_summary={}
    )
    final_state = await app.ainvoke(initial_state)
    return final_state['final_summary']

In [65]:
import json

# Run the summarization
summary_data = await run_summarization([Document(page_content="test")])
# Print the final summary data

print(json.dumps(summary_data, indent=2))

with open("Output.json", "w") as f:
    f.write(summary_data)



ValidationError: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=Document(metadata={}, page_content='test'), input_type=Document]
    For further information visit https://errors.pydantic.dev/2.9/v/string_type